# Generic XGBoost (XGB) Model

This notebook provides an enhanced generic XGB modeling framework that can handle any dataset with categorical variables (3-10 categories) without knowing specific variable names.

## Features
- **Enhanced target/ID detection**: Supports `var1` and other common variable names
- **Automatic detection of categorical dummy variables**
- **Generic handling of any variable names**
- **Automatic reference column selection for categorical variables**
- **Flexible data loading and preprocessing**
- **Comprehensive model training and evaluation**
- **Configurable model parameters**

## Configuration Options
- **Target variable**: Automatically detects `var1` or other common names
- **ID variable**: Automatically detects common ID column patterns
- **Model parameters**: Configurable hyperparameters matching original notebooks
- **Data prefix**: Flexible dataset naming

## Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Set CPU fallback for MPS (Apple Silicon)
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
print("PYTORCH_ENABLE_MPS_FALLBACK =", os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import logging
import xgboost as xgb
from pathlib import Path

from datetime import datetime
from prism.config import INTERIM_DATA_DIR, MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT
from prism.config import DATASET_PREFIX, CONFIG_NAME
from prism.logging_config import setup_logging
from prism.config_loader import (
    load_config_if_exists,
    detect_target_and_id_columns,
    DEFAULT_TARGET_CANDIDATES,
    DEFAULT_ID_CANDIDATES,
)

# =============================================================================
# DATASET CHECK - Ensure .env is configured
# =============================================================================
if DATASET_PREFIX is None:
    print("=" * 70)
    print("ERROR: No dataset configured!")
    print("=" * 70)
    print("\nPlease configure your dataset in the .env file:")
    print("  1. Copy .env.example to .env")
    print("  2. Set PRISM_CONFIG=your_config (loads example_notebooks/config/your_config.yaml)")
    print("     OR set PRISM_DATASET=your_dataset (quick mode, no YAML needed)")
    print("\nExample .env contents:")
    print("  PRISM_CONFIG=htx_example")
    print("=" * 70)
    raise ValueError("DATASET_PREFIX is None - configure .env file")

print(f"Dataset: {DATASET_PREFIX}")
if CONFIG_NAME:
    print(f"Config: {CONFIG_NAME}.yaml")

from prism.device_tools import get_device, get_num_cpu_workers, to_xgb_device
from prism.preprocessing import PRiSMScaler, detect_reference_columns
from prism.data_utilities import save_split_datasets
from prism.metrics import evaluate_model_performance
from prism.wrapper import SklearnWrapper

## Configuration


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
model_name = 'xgb'
DATA_PREFIX = DATASET_PREFIX  # from config.py

# Random seed for reproducibility
random_seed = 257

# ============================================================================
# TARGET AND ID VARIABLE CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
# These defaults are imported from prism.config_loader but can be overridden here.
# To customize, uncomment and modify the lines below:
# TARGET_CANDIDATES = ['my_target', 'outcome', 'label']
# ID_CANDIDATES = ['my_id', 'patient_id']
TARGET_CANDIDATES = list(DEFAULT_TARGET_CANDIDATES)  # Copy to allow modification
ID_CANDIDATES = list(DEFAULT_ID_CANDIDATES)  # Copy to allow modification

# ============================================================================
# LOAD CONFIG FILE (OVERRIDES MANUAL SETTINGS ABOVE)
# ============================================================================
# Use CONFIG_NAME if set (from .env PRISM_CONFIG), otherwise try DATA_PREFIX
config_to_load = CONFIG_NAME if CONFIG_NAME else DATA_PREFIX
config = load_config_if_exists(config_to_load)

if config:
    print(f"Config file found: {config_to_load}.yaml")
    if 'random_seed' in config:
        random_seed = config['random_seed']
        print(f"   [ok] random_seed: {random_seed}")
    if 'target_candidates' in config:
        TARGET_CANDIDATES = config['target_candidates']
        print(f"   [ok] target_candidates: {TARGET_CANDIDATES}")
    if 'id_candidates' in config:
        ID_CANDIDATES = config['id_candidates']
        print(f"   [ok] id_candidates: {ID_CANDIDATES}")
else:
    print("No config file found - using manual settings above")
    print("Edit the settings in the cells above if needed.")

# Apply random seed
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)

# Device: 'device' in the YAML config or the PRISM_DEVICE env var overrides
# auto-detection ('auto', 'cpu', 'cuda', 'mps'); default is the best available
device = get_device(config.get('device') if config else None)
print(f"Using device: {device}")

# Check the data prefix that will be used to load train, test, and validation cohorts
print(f"DATA_PREFIX = '{DATA_PREFIX}'")
print(f"Will load data files from {INTERIM_DATA_DIR}:")

# Define filename variables
train_filename = f'{DATA_PREFIX}_train.csv'
test_filename = f'{DATA_PREFIX}_test.csv'
val_filename = f'{DATA_PREFIX}_val.csv'

print(f"  - {train_filename}")
print(f"  - {test_filename}") 
print(f"  - {val_filename}")

model_identifier = f"{DATA_PREFIX}_{model_name}"
print(f"Model identifier: {model_identifier}")

# Set up logging (use logging.DEBUG for more detailed logs, logging.INFO for less detailed logs)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
setup_logging(
    file_log_level=logging.INFO,
    log_file=f'model_{DATA_PREFIX}_{model_identifier}_{timestamp}.log',
    console_output=True,
    root_log_level=logging.WARNING,
)

## Data Loading and Variable Detection


In [ ]:
# Load data
print("Loading data...")
# Skip comment lines that start with '#' to avoid header issues
data_train = pd.read_csv(INTERIM_DATA_DIR.joinpath(train_filename), comment='#')
data_test = pd.read_csv(INTERIM_DATA_DIR.joinpath(test_filename), comment='#')
data_val = pd.read_csv(INTERIM_DATA_DIR.joinpath(val_filename), comment='#')

print(f"Training data shape: {data_train.shape}")
print(f"Test data shape: {data_test.shape}")
print(f"Validation data shape: {data_val.shape}")

In [ ]:
# Detect target and ID columns using the centralized function from prism.config_loader
target_column, id_column = detect_target_and_id_columns(
    data_train,
    target_candidates=TARGET_CANDIDATES,
    id_candidates=ID_CANDIDATES,
)

# Fallback detection if automatic detection fails
if target_column is None:
    print("\n" + "=" * 50)
    print("FALLBACK TARGET DETECTION")
    print("=" * 50)

    # Try to find binary columns that could be targets
    binary_cols = []
    for col in data_train.columns:
        unique_vals = sorted(data_train[col].dropna().unique())
        if len(unique_vals) == 2 and set(unique_vals) == {0, 1}:
            binary_cols.append(col)

    if binary_cols:
        target_column = binary_cols[0]  # Use first binary column
        print(f"Found binary columns: {binary_cols}")
        print(f"Using as target: {target_column}")
    else:
        # Last resort: use first column
        target_column = data_train.columns[0]
        print(f"No binary columns found. Using first column as target: {target_column}")

if id_column is None:
    print("\n" + "=" * 50)
    print("FALLBACK ID DETECTION")
    print("=" * 50)
    print("No ID column found. Will create synthetic IDs during data preparation.")

print(f"\nFinal detection results:")
print(f"  Target column: {target_column}")
print(f"  ID column: {id_column}")

In [ ]:
# Note: Reference columns are now dropped during preprocessing (drop_reference_columns=True)
# This cell shows reference column info from the config loaded earlier

if config:
    reference_column_strategy = config.get('reference_column_strategy', 'alphabetical')
    manual_reference_columns = config.get('reference_columns', None)

    print(f"Reference column strategy: {reference_column_strategy}")
    if manual_reference_columns:
        print(f"Reference columns (already dropped during preprocessing):")
        for group, ref_col in manual_reference_columns.items():
            print(f"  {group}: {ref_col}")
else:
    print("No config file - using default reference column detection")

print("\n" + "=" * 70)
print("NOTE: Reference columns have been dropped during preprocessing.")
print("All one-hot encoded groups now have N-1 columns (reference excluded).")
print("=" * 70)

## Data Preparation


In [ ]:
# Prepare data for modeling
print("Preparing data for modeling...")

# If target column is not detected, try to find it manually
if target_column is None:
    print("\nWARNING: Target column not automatically detected!")
    print("Available columns:", list(data_train.columns))
    
    # Try to find the first column that looks like a target (binary 0/1)
    for col in data_train.columns:
        unique_vals = sorted(data_train[col].dropna().unique())
        if len(unique_vals) == 2 and set(unique_vals) == {0, 1}:
            target_column = col
            print(f"  Found potential target column: {col} with values {unique_vals}")
            break
    
    if target_column is None:
        # If still not found, use the first column as target
        target_column = data_train.columns[0]
        print(f"  Using first column as target: {target_column}")
        print(f"  Values: {sorted(data_train[target_column].dropna().unique())}")

# If ID column is not detected, create a synthetic one
if id_column is None:
    print("\nWARNING: ID column not automatically detected!")
    id_column = 'trr_id_code'
    print(f"  Creating synthetic ID column: {id_column}")
    
    # Add synthetic IDs to all datasets
    for i, df in enumerate([data_train, data_test, data_val]):
        df[id_column] = [f"B{j:06d}" for j in range(len(df))]
    print(f"  Added synthetic IDs to all datasets")

# Define columns to drop (just target and ID - reference columns already dropped)
drop_cols = [target_column, id_column]

print(f"\nColumns to drop:")
print(f"  Target: {target_column}")
print(f"  ID: {id_column}")
print(f"  (Reference columns were already dropped during preprocessing)")

# Prepare feature matrices
X_train = data_train.drop(drop_cols, axis=1)
y_train = data_train[target_column]

X_test = data_test.drop(drop_cols, axis=1)
y_test = data_test[target_column]

print(f"\nFeature matrices prepared:")
print(f"  Training: {X_train.shape}")
print(f"  Test: {X_test.shape}")

# Get feature names (as in original notebook)
feature_names = X_train.columns.tolist()

# Check class distribution
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

print(f"\nClass distribution in test set:")
print(y_test.value_counts(normalize=True))

## Data Scaling and Preprocessing


In [ ]:
# Create and fit PRiSMScaler on training data
# PRiSMScaler uses MedianStdScaler with sd_scale=2.0 by default
scaler = PRiSMScaler(scaler='median_std')
scaler.fit(X_train)

# Transform data and convert to tensors using to_tensor method
X_train_tensor = scaler.to_tensor(X_train, device=device)
X_test_tensor = scaler.to_tensor(X_test, device=device)

# Convert target variables to tensors
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32, device=device)

print(f"Scaler type: {type(scaler).__name__}")
print(f"Underlying scaler: {type(scaler.scaler).__name__}")
print(f"Training data shape: {X_train_tensor.shape}")
print(f"Training tensor device: {X_train_tensor.device}")

## Hyperparameter Tuning (Optional)

Hyperparameters are determined in this order:
1. If `hyperparameter_tuning.xgb.params_file` is specified -> Load from that file
2. If `hyperparameter_tuning.xgb.enabled: true` -> Run Optuna tuning now
3. If `hyperparameter_tuning.skip_saved_params: true` -> Use hardcoded defaults
4. If a best_params file exists in MODELS_DIR -> Load from there
5. Otherwise -> Use hardcoded defaults

The --no-tune flag (sets PRISM_NO_TUNE env var) skips inline tuning.

In [ ]:
# Hyperparameter tuning or loading
import os
from prism.hyperparameter_tuning import (
    load_best_params, load_params_from_file, run_hyperparameter_tuning, save_best_params,
    save_best_model, load_tuned_model
)
from prism.config_loader import get_tuning_config

# Get tuning config
tuning_config = get_tuning_config(config, 'xgb')

# Check if tuning should be skipped via environment variable (set by --no-tune flag)
no_tune = os.environ.get('PRISM_NO_TUNE', '').lower() in ('1', 'true', 'yes')

tuned_params = None
best_tuned_model = None  # Track best model from tuning
if hasattr(tuning_config, 'params_file') and tuning_config.params_file:
    # Option 1: Explicit params_file specified in config
    print(f"Loading parameters from config-specified file: {tuning_config.params_file}")
    tuned_params = load_params_from_file(tuning_config.params_file)
elif tuning_config.enabled and not no_tune:
    # Option 2: Run hyperparameter tuning now
    # Use scaled numpy arrays for XGBoost (not tensors)
    print(f"Running XGBoost hyperparameter tuning ({tuning_config.n_trials} trials)...")
    tuned_params, study, best_tuned_model = run_hyperparameter_tuning(
        model_type='xgb',
        X_train=scaler.transform(X_train),
        y_train=y_train.values,
        X_test=scaler.transform(X_test),
        y_test=y_test.values,
        tuning_config=tuning_config,
        random_seed=random_seed,
        device=str(device)  # GPU acceleration for XGBoost when CUDA available
    )
    # Save the results
    save_best_params(tuned_params, 'xgb', DATA_PREFIX, MODELS_DIR, study)
    print(f"Tuning complete. Best AUC: {study.best_value:.4f}")
    print(f"Best params saved to {MODELS_DIR}")
    # Save the best model from tuning
    if best_tuned_model is not None:
        save_best_model(
            model=best_tuned_model,
            model_type='xgb',
            dataset_prefix=model_identifier,
            output_dir=MODELS_DIR,
            scaler=scaler,
            hyperparameters=tuned_params,
            feature_names=feature_names
        )
        print(f"Best tuned model saved to {MODELS_DIR / model_identifier}")
elif getattr(tuning_config, 'skip_saved_params', False):
    # Option 3a: skip_saved_params=True -> use defaults only
    print("skip_saved_params=True, using default hyperparameters.")
else:
    # Option 3b: Try to find existing params in MODELS_DIR
    tuned_params = load_best_params('xgb', DATA_PREFIX, MODELS_DIR)
    if tuned_params:
        print(f"Loaded existing tuned params from {MODELS_DIR}")

if tuned_params:
    print("Using hyperparameters:")
    for key, value in tuned_params.items():
        print(f"  {key}: {value}")
else:
    print("No tuned hyperparameters found. Using defaults.")

## Model Training

In [ ]:
# Check if a tuned model already exists (skip training if so)
tuned_model_checkpoint = load_tuned_model('xgb', model_identifier, MODELS_DIR)
xgb_model = None

if tuned_model_checkpoint is not None:
    print(f"Found tuned model - loading instead of retraining...")
    xgb_model = tuned_model_checkpoint['model']
    # Use hyperparameters from the tuned model
    xgb_hyperparameters = tuned_model_checkpoint.get('hyperparameters', {})
    print(f"Loaded tuned model: {type(xgb_model).__name__}")
    print(f"Hyperparameters from tuned model: {xgb_hyperparameters}")
else:
    # XGBoost hyperparameters - use tuned params if available, otherwise defaults
    xgb_hyperparameters = {
        'objective': 'binary:logistic',  # Objective function
        'eval_metric': 'logloss',
        'max_depth': tuned_params.get('max_depth', 4) if tuned_params else 4,
        'learning_rate': tuned_params.get('learning_rate', 0.055) if tuned_params else 0.055,
        'subsample': tuned_params.get('subsample', 0.75) if tuned_params else 0.75,
        'colsample_bytree': tuned_params.get('colsample_bytree', 0.65) if tuned_params else 0.65,
        'min_child_weight': tuned_params.get('min_child_weight', 9) if tuned_params else 9,
        'n_estimators': tuned_params.get('n_estimators', 280) if tuned_params else 280,
        'random_state': random_seed,  # Random seed for reproducibility
        'early_stopping_rounds': tuned_params.get('early_stopping_rounds', 10) if tuned_params else 10,
        'device': to_xgb_device(device),  # GPU acceleration when CUDA available
    }

    print(f"XGBoost device: {xgb_hyperparameters['device']} (from PyTorch device: {device})")
    print(f"Training XGBoost with {X_train.shape[1]} input features...")
    xgb_model = xgb.XGBClassifier(**xgb_hyperparameters)

    # Convert tensors to numpy arrays for XGBoost
    X_train_np = X_train_tensor.cpu().numpy()
    X_test_np = X_test_tensor.cpu().numpy()
    y_train_np = y_train.values
    y_test_np = y_test.values

    # Create evaluation set (train + test for tracking both loss curves)
    eval_set = [(X_train_np, y_train_np), (X_test_np, y_test_np)]
    # Train the model and store evaluation results
    xgb_model.fit(
        X_train_np,
        y_train_np,
        eval_set=eval_set,
        verbose=False
    )

In [ ]:
# Plot training loss curve
from prism.notebook_utils import get_training_history, plot_training_history

if hasattr(xgb_model, 'best_iteration'):
    print(f"Best iteration: {xgb_model.best_iteration}")

training_history = get_training_history(xgb_model, tuned_model_checkpoint)
plot_training_history(training_history, title_prefix='XGBoost', x_label='Boosting Round')

## Model Evaluation

In [ ]:
model = SklearnWrapper(xgb_model)

In [ ]:
# Evaluate model performance
print("Evaluating model performance...")

# Training set evaluation
print("Evaluating on training set...")
y_pred_train = model.predict(X_train_tensor)
train_metrics = evaluate_model_performance(
    y_train_tensor,
    y_pred_train,
    y_train_tensor,
    title=f"{model_name} - Training Data",
    random_seed=random_seed,
)

# Test set evaluation
print("Evaluating on test set...")
y_pred_test = model.predict(X_test_tensor)
test_metrics = evaluate_model_performance(
    y_test,
    y_pred_test,
    y_train,
    title=f"{model_name} - Test Data",
    random_seed=random_seed,
)

print("\nModel Performance Summary:")
print("=" * 50)
print(f"Training - AUROC: {train_metrics['auroc']:.4f}, Accuracy: {train_metrics['accuracy']:.4f}")
print(f"Test     - AUROC: {test_metrics['auroc']:.4f}, Accuracy: {test_metrics['accuracy']:.4f}")

## Model Saving

In [ ]:
# Save the model and scaler
print("Saving model and scaler...")
# Create timestamp for unique filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Gather hyperparameters
hyperparameters = xgb_hyperparameters

# Set filepath
model_dir = MODELS_DIR.joinpath(model_identifier)
os.makedirs(model_dir, exist_ok=True)

model_filename = f"{model_identifier}_model_{timestamp}.pt"
model_path = model_dir.joinpath(model_filename)

# Create save dict with standard items
save_dict = {
    'scaler': scaler,
    'model': model,  # Full model for convenience
    'model_class': type(model),
    'hyperparameters': hyperparameters,
    'feature_names': feature_names,
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
}

# If the model has a state_dict method (PyTorch models), save it
if hasattr(model, 'state_dict') and callable(getattr(model, 'state_dict')):
    save_dict['model_state_dict'] = model.state_dict()

# If the model is a wrapper, also save the underlying model class
if hasattr(model, 'model'):
    save_dict['underlying_model'] = model.model
    save_dict['underlying_model_class'] = type(model.model)

# Save the model in PyTorch standard format
torch.save(save_dict, model_path)

print(f"Model saved in standard PyTorch format to {model_path}")

In [ ]:
# Save processed data to PROCESSED_DATA_DIR
# Note: Reference columns have already been dropped during preprocessing
print("\nSaving processed data...")

save_split_datasets(
    train_df=data_train,
    test_df=data_test,
    val_df=data_val,
    base_filename=model_identifier,
    save_dir=PROCESSED_DATA_DIR,
    include_timestamp=True,
    header_comment=f"target: {target_column}, patient IDs: {id_column}. All other features are model inputs, pre-normalization. Reference columns were dropped during preprocessing.",
)

print("Processed data saved successfully.")

In [ ]:
print(f"\n" + "=" * 60)
print("MODEL TRAINING COMPLETE")
print("=" * 60)
print(f"Model identifier: {model_identifier}")
print(f"Model saved to: {model_path}")
print(f"Processed data saved to: {PROCESSED_DATA_DIR}")
print(f"Final test AUROC: {test_metrics['auroc']:.4f}")
print(f"Final test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Features used: {len(feature_names)}")
print(f"Target variable: {target_column}")
print(f"ID variable: {id_column}")